In [1]:
import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import ScalarMappable

# --------------------------
# 1) 参数设置 - 径向动量收支诊断
# --------------------------
nc_file = "dataset/typhoon_azimuthal_avg_budget.nc"
target_time = 3600*12
max_r_km = 300.0
max_z_km = 20.0

# 诊断模式：此单元为径向模式
diagnosis_mode = 'radical'  # 目前仅支持切向模式，后续可扩展为径向模式
base_var = 'ur'
budget_prefix = 'br_'

# 离散色阶级数
n_levels_u = 17
n_levels_budget = 17

# 每页最多展示变量数（单页模式下仅保留参数，不再分页）
panels_per_page = 10

# 预算项优先顺序（使用后缀，如 "hadv" 代表 br_hadv）
preferred_budget_order = [
    "hadv", "vadv",
    "hidiff", "vidiff",
    "hturb", "vturb",
    "pgrad", "cor", "rdamp",
]

# --------------------------
# 2) 读数据
# --------------------------
with Dataset(nc_file, "r") as nc:
    time_arr = nc.variables["time"][:]
    r_arr = nc.variables["r"][:]

    if "z" in nc.variables:
        z_arr = nc.variables["z"][:]
    elif "zh" in nc.variables:
        z_arr = nc.variables["zh"][:]
    else:
        raise KeyError("未找到 z 或 zh 坐标变量。")

    t_idx = int(np.argmin(np.abs(time_arr - target_time)))
    actual_time = float(time_arr[t_idx])
    print(f"诊断模式: 径向动量收支诊断 (ur + br_*)")
    print(f"目标时间: {target_time:.1f}s, 实际时间: {actual_time:.1f}s, 索引: {t_idx}")

    r_mask = r_arr <= max_r_km
    z_mask = z_arr <= max_z_km
    r_plot = r_arr[r_mask]
    z_plot = z_arr[z_mask]

    var_names = list(nc.variables.keys())
    budget_vars = [v for v in var_names if v.startswith(budget_prefix)]

    # 按优先级排序预算项
    budget_suffixes = [v[len(budget_prefix):] for v in budget_vars]
    ordered_suffixes = [s for s in preferred_budget_order if s in budget_suffixes]
    ordered_suffixes += [s for s in budget_suffixes if s not in ordered_suffixes]
    ordered_budget = [budget_prefix + s for s in ordered_suffixes]

    plot_vars = [base_var] + ordered_budget

    field_data = {}
    field_units = {}
    field_long_name = {}

    for vname in plot_vars:
        if vname not in nc.variables:
            print(f"警告: 变量 {vname} 不在文件中")
            continue
        v = nc.variables[vname]
        field_data[vname] = np.asarray(v[t_idx, z_mask, r_mask], dtype=float)
        field_units[vname] = getattr(v, "units", "")
        field_long_name[vname] = getattr(v, "long_name", vname)

if base_var not in field_data:
    raise KeyError(f"文件中未找到变量 {base_var}。")

budget_names = [v for v in plot_vars if v != base_var and v in field_data]

# 新增：所有收支项求和子图
budget_sum_name = f"{budget_prefix}sum"
if len(budget_names) > 0:
    field_data[budget_sum_name] = np.sum(np.stack([field_data[v] for v in budget_names], axis=0), axis=0)
    field_units[budget_sum_name] = field_units.get(budget_names[0], "")
    field_long_name[budget_sum_name] = "Sum of all budget terms"

all_panels = [base_var]
if budget_sum_name in field_data:
    all_panels.append(budget_sum_name)
all_panels += budget_names
print(f"将绘制变量数: {len(all_panels)} ({base_var} + budget_sum + {len(budget_names)} 个收支项)")

# --------------------------
# 3) 设置统一色阶（离散）
# --------------------------
u_data = field_data[base_var]
u_abs = np.nanpercentile(np.abs(u_data), 99.0)
u_abs = max(float(u_abs), 1e-8)
u_levels = np.linspace(-u_abs, u_abs, n_levels_u)
u_norm = BoundaryNorm(u_levels, ncolors=256, clip=True)

budget_scale_vars = list(budget_names)
if budget_sum_name in field_data:
    budget_scale_vars.append(budget_sum_name)

if len(budget_scale_vars) > 0:
    b_stack = np.stack([field_data[v] for v in budget_scale_vars], axis=0)
    b_abs = np.nanpercentile(np.abs(b_stack), 99.0)
    b_abs = max(float(b_abs), 1e-12)
else:
    b_abs = 1e-12

b_levels = np.linspace(-b_abs, b_abs, n_levels_budget)
b_norm = BoundaryNorm(b_levels, ncolors=256, clip=True)

print(f"ur 范围: [{-u_abs:.2f}, {u_abs:.2f}]")
print(f"预算项范围(含sum): [{-b_abs:.6f}, {b_abs:.6f}]")

# --------------------------
# 4) 单页绘图（自适应网格）
# --------------------------
R, Z = np.meshgrid(r_plot, z_plot)

ncols = 3
n_panels = len(all_panels)
nrows = int(np.ceil(n_panels / ncols))
fig_h = 4.2 * nrows + 2.6

fig, axes = plt.subplots(nrows, ncols, figsize=(18, fig_h), sharex=True, sharey=True)
axes_flat = np.atleast_1d(axes).ravel()

for i, vname in enumerate(all_panels):
    ax = axes_flat[i]
    data = field_data[vname]

    if vname == base_var:
        levels = u_levels
        norm = u_norm
    else:
        levels = b_levels
        norm = b_norm

    cf = ax.contourf(
        R, Z, data,
        levels=levels,
        cmap="RdBu_r",
        norm=norm,
        extend="both"
    )

    c_levels = levels[::2]
    c_levels = c_levels[~np.isclose(c_levels, 0.0, atol=1e-15)]
    if c_levels.size > 0:
        cs = ax.contour(R, Z, data, levels=c_levels, colors="k", linewidths=0.7, alpha=0.75)
        ax.clabel(cs, inline=True, fontsize=8, fmt="%.2g")

    long_name = field_long_name.get(vname, vname)
    unit = field_units.get(vname, "")
    if unit:
        ax.set_title(f"{vname} ({unit})\n{long_name}", fontsize=10)
    else:
        ax.set_title(f"{vname}\n{long_name}", fontsize=10)

    ax.grid(True, linestyle="--", alpha=0.25)
    ax.set_xlim(0, max_r_km)
    ax.set_ylim(0, max_z_km)

for idx, ax in enumerate(axes_flat):
    if idx >= n_panels:
        ax.axis("off")
        continue
    if idx % ncols == 0:
        ax.set_ylabel("Height (km)", fontsize=12)
    if idx >= (nrows - 1) * ncols:
        ax.set_xlabel("Radius (km)", fontsize=12)

# 统一把说明文字和色标放到图外下方，避免遮挡任意子图
sm_u = ScalarMappable(norm=u_norm, cmap="RdBu_r")
sm_u.set_array([])
sm_b = ScalarMappable(norm=b_norm, cmap="RdBu_r")
sm_b.set_array([])

fig.suptitle("Radial Momentum Budget Azimuthal-Mean Sections (Single Page)", fontsize=16, y=0.985)
fig.subplots_adjust(left=0.06, right=0.97, bottom=0.12, top=0.95, wspace=0.16, hspace=0.22)

info_text = [
    "Azimuthal-Mean Radial Momentum Budget Diagnostics",
    f"Time: {actual_time:.1f} s",
    f"Radius: 0-{max_r_km:.0f} km",
    f"Height: 0-{max_z_km:.0f} km",
    "Data: dataset/typhoon_azimuthal_avg_budget.nc",
    "Shading: discrete levels; lines: contour labels",
]
fig.text(0.06, 0.085, "\n".join(info_text), va="top", ha="left", fontsize=10)

cax1 = fig.add_axes([0.58, 0.075, 0.35, 0.015])
cax2 = fig.add_axes([0.58, 0.04, 0.35, 0.015])

cb1 = fig.colorbar(sm_u, cax=cax1, orientation="horizontal", ticks=u_levels[::2])
cb1.set_label(f"{base_var} ({field_units.get(base_var, '')})", fontsize=10)

budget_unit = field_units.get(budget_names[0], "") if len(budget_names) > 0 else ""
cb2 = fig.colorbar(sm_b, cax=cax2, orientation="horizontal", ticks=b_levels[::2])
cb2.set_label(f"radial budget terms + sum ({budget_unit})", fontsize=10)

plt.show()

print("径向诊断绘图完成！")

OSError: [Errno -51] NetCDF: Unknown file format: 'dataset/typhoon_azimuthal_avg_budget.nc'

## 使用说明

本 Notebook 针对柱坐标下的方位角平均数据，提供 **径向** 和 **切向** 两种动量收支诊断模式：

### 径向诊断（第一个代码单元）
- **读取变量**：`ur`（径向风）及 `br_*`（径向收支项）
- **色阶**：统一的对称范围（0 为中心），RdBu_r 配色
- **适用场景**：分析径向流入、辐散、水平平流等效应

### 切向诊断（第二个代码单元）
- **读取变量**：`ut`（切向风，气旋性风速为正）及 `bt_*`（切向收支项）
- **色阶**：同上，对称范围
- **适用场景**：分析切向加速、摩擦层影响、科氏力作用

### 快速切换方法
在任一代码单元中修改：
```python
diagnosis_mode = 'radial'      # 径向诊断
# 或
diagnosis_mode = 'tangential'  # 切向诊断
```

### 自定义参数
- `target_time`：目标时间（秒）
- `max_r_km`、`max_z_km`：显示范围
- `n_levels_u`、`n_levels_budget`：色阶级数
- `panels_per_page`：每页显示的变量数（4×3 布局）

### 输出说明
- **风场面板**（第一列）：显示绝对值范围
- **收支项面板**（后续列）：使用相同的对称色阶，便于对比
- **信息面板**：显示模式、时间、坐标范围等关键信息